# PETRA + T1 Integration for Segmentation and Labeling

In [ ]:
import nibabel as nib
import numpy as np
import ants

In [ ]:
# --- Step 1: Load Images ---
t1_path = "T1w_image.nii.gz"  # Replace with your T1 path
petra_path = "PETRA_image.nii.gz"  # Replace with your PETRA path
t1 = ants.image_read(t1_path)
petra = ants.image_read(petra_path)

In [ ]:
# --- Step 2: Register PETRA to T1 ---
reg = ants.registration(fixed=t1, moving=petra, type_of_transform='Rigid')
petra_reg = reg['warpedmovout']
registered_petra_path = "PETRA_registered.nii.gz"
ants.image_write(petra_reg, registered_petra_path)

In [ ]:
# --- Step 3: Load Registered PETRA with nibabel for labeling ---
petra_img = nib.load(registered_petra_path)
petra_data = petra_img.get_fdata()
# Threshold PETRA to identify bone/skull
bone_threshold = 100  # Adjust as needed
bone_mask = (petra_data > bone_threshold).astype(np.uint8) * 6  # Label 6 = bone

In [ ]:
# --- Step 4: Load existing label map ---
label_map_path = "label_map_ras_clean.nii.gz"  # Adjust path
label_img = nib.load(label_map_path)
label_map = label_img.get_fdata().astype(np.uint8)

In [ ]:
# Combine labels: insert bone where label_map == 0
combined_label = np.where(label_map == 0, bone_mask, label_map)

In [ ]:
# Save combined label map
combined_img = nib.Nifti1Image(combined_label, affine=label_img.affine, header=label_img.header)
nib.save(combined_img, "label_map_with_petra.nii.gz")